# Real-data training and validation

Use your authorized local `data/icu_train.csv`. Records and generated model files stay local. Install this project with the `notebook` extras and select that Python environment. This workflow reports an internal validation split, not an external clinical test.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
from icu_mortality.train import train
from icu_mortality.report import create_report

project = Path.cwd()
if project.name == 'notebooks':
    project = project.parent
if not (project / 'pyproject.toml').exists():
    raise RuntimeError('Start Jupyter from the ICU project directory or its notebooks folder.')
data = project / 'data/icu_train.csv'
if not data.exists():
    raise FileNotFoundError('Place your authorized icu_train.csv in data/, or use quickstart.ipynb for the synthetic demo.')
run = project / 'artifacts' / ('real-notebook-' + uuid4().hex[:8])
metrics = train(data, project / 'configs/default.json', run)
print('Internal validation only')
print(json.dumps(metrics, indent=2))


## Aggregate validation diagnostics

The report reconstructs the recorded split and verifies the training CSV checksum. No patient rows or identifiers are displayed. Class weighting and threshold choice affect the precision/recall tradeoff.


In [ ]:
report = create_report(run, data, run / 'report')
from IPython.display import Image, display
display(Image(filename=str(run / 'report/validation-curves.png')))
display(Image(filename=str(run / 'report/confusion-matrix.png')))


## Optional inference on a separate file

This cell only writes local predictions. It does not evaluate hidden-test labels or assume their row order.


In [ ]:
from icu_mortality.predict import predict
hidden = project / 'data/icu_hidden_test.csv'
if hidden.exists():
    predictions = predict(run / 'pipeline.joblib', hidden, run / 'hidden-test-predictions.csv')
    print('Predictions written for', len(predictions), 'rows to', run)
else:
    print('Optional hidden-test CSV not supplied; training and validation are complete.')


## Interpretation and provenance

See `docs/real-data-results.md`, `docs/data-provenance.md`, and `MODEL_CARD.md`. Confirm upstream data identity, permitted use, feature timing, and an appropriate external validation design before broader claims.
